建立鄰接矩陣，最大距離為100KM(可調整)

In [ ]:
"""
建立 166×166 鄰接矩陣 adj.npy
- 使用 all_station_info.json 中的經緯度計算測站間的實際距離（km）
- 距離 ≤ D_threshold 的測站之間有連接，權重 = 1 - D_ij / D_max
- 距離 > D_threshold 的設為 0（無連接）
- 對角線設為 1（自連接）
"""

import json
import numpy as np
from pathlib import Path
from math import radians, sin, cos, sqrt, atan2

# ============ 設定區 ============
ROOT_DIR = Path(r"D:\filtered_by_Hualien_events_extracted_waveforms")

STATION_JSON = r"C:\Users\user\Desktop\L_Earthquake_Model\earthquake_project\source\all_stastion_info.json"
OUTPUT_PATH = ROOT_DIR / "adj.npy"
D_THRESHOLD_KM = 100  # ← 距離閾值（公里），超過此距離視為無連接
# ================================


def haversine_km(lat1, lon1, lat2, lon2):
    """
    使用 Haversine 公式計算兩點之間的距離（公里）。
    比直接用經緯度算歐氏距離更準確，因為考慮了地球曲率。
    """
    R = 6371.0  # 地球平均半徑（公里）
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c


def main():
    # 1. 讀取測站座標
    with open(STATION_JSON, "r", encoding="utf-8") as f:
        station_data = json.load(f)

    stations = list(station_data.keys())
    coords = np.array(list(station_data.values()))  # [166, 2]  (lat, lon)
    n = len(stations)
    print(f"📋 測站數量：{n}")
    print(f"📏 距離閾值：{D_THRESHOLD_KM} km")

    # 2. 計算距離矩陣（公里）
    dist_matrix = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        for j in range(i + 1, n):
            d = haversine_km(coords[i, 0], coords[i, 1],
                             coords[j, 0], coords[j, 1])
            dist_matrix[i, j] = d
            dist_matrix[j, i] = d

    print(f"📐 距離範圍：{dist_matrix[dist_matrix > 0].min():.1f} ~ {dist_matrix.max():.1f} km")

    # 3. 建立鄰接矩陣
    #    A_ij = 1 - D_ij / D_max（在閾值內的）
    #    A_ij = 0（超過閾值的）
    #    D_max = 閾值內的最大距離（即 D_THRESHOLD_KM）
    adj = np.zeros((n, n), dtype=np.float32)

    D_max = D_THRESHOLD_KM
    connected_count = 0

    for i in range(n):
        for j in range(n):
            if i == j:
                adj[i, j] = 1.0  # 自連接
            elif dist_matrix[i, j] <= D_THRESHOLD_KM:
                adj[i, j] = 1.0 - dist_matrix[i, j] / D_max
                connected_count += 1

    total_pairs = n * (n - 1)
    connectivity = connected_count / total_pairs * 100
    avg_neighbors = connected_count / n

    print(f"\n📊 鄰接矩陣統計：")
    print(f"   矩陣大小：[{n}, {n}]")
    print(f"   有連接的邊數：{connected_count}（{connectivity:.1f}%）")
    print(f"   平均每站鄰居數：{avg_neighbors:.1f}")
    print(f"   權重範圍：{adj[adj > 0].min():.4f} ~ {adj.max():.4f}")

    # 4. 儲存
    np.save(OUTPUT_PATH, adj)
    print(f"\n✅ 已儲存：{OUTPUT_PATH}")
    print(f"   shape = {adj.shape}")
    print("🎉 完成！")

    # 5. 印出幾個範例
    print(f"\n📌 範例（前 5 站的鄰居數）：")
    for i in range(min(5, n)):
        neighbors = np.sum(adj[i] > 0) - 1  # 扣掉自己
        print(f"   {stations[i]}：{int(neighbors)} 個鄰居")


if __name__ == "__main__":
    main()

FileNotFoundError: [Errno 2] No such file or directory: './source/all_stastion_info.json'